# Transformador de decisão SRA: análise de roteamento

Nesta demonstração, usamos o SRA como um **Transformador de Decisão (DT)** para aprendizado por reforço e analisamos como o agente "usa diferentes partes de seu cérebro" ao resolver diferentes tarefas do GridWorld (labirinto) (caça ao tesouro, fuga, etc.).

Visualizamos os dois pontos a seguir:
1. **Separação de tarefas**: preconceito em que Sinapses são usadas para "Escape" versus "Tesouro".
2. **Separação de percepção e ação**: como a Sinapse responsável muda dependendo do tipo de token alimentado: "Estado", "Recompensa" ou "Ação".

## 1. Configuração do ambiente

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')


## 2. Prepare o modelo e as tarefas
Definimos um modelo com 16 especialistas (Sinapses).

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.sra_language_models import MoESRALanguageModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_gridworld import generate_trajectory, make_dt_batch
from src.sra_experiment import make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

config = MoESRAConfig(
    vocab_size=100,
    d_model=64,
    n_layers=2,
    n_heads=2,
    num_synapses=16, # 16 experts ready
    k=2,
    max_seq_len=64
)
model = MoESRALanguageModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=128, max_seq_len=200, pad_idx=0).to(device)
optimizer = make_optimizer(model, lr=0.005)

## 3. Mini ciclo de treinamento
Geramos aleatoriamente trajetórias das tarefas `treasure` e `escape` e treinamos em apenas 50 etapas. (Mantenha-se breve para que termine rapidamente no Colab, etc.)

In [ ]:
print("Training Decision Transformer...")
model.train()

epochs = 150
batch_size = 32
max_steps = 10

for epoch in range(epochs):
    x, y, _ = make_dt_batch(batch_size, max_steps, device)
    
    optimizer.zero_grad()
    outputs, routing_weights = model(x)
    
    # Compute loss only on action prediction
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1), ignore_index=-100)
    
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("Training finished!")

## 4. Análise de roteamento: uso de sinapse por tarefa
Após o treinamento, agregamos e comparamos quais sinapses o roteador prefere usar para “caça ao tesouro” versus “fuga”.

In [ ]:
def analyze_task_usage(task_type, samples=50):
    model.eval()
    usage_counts = torch.zeros(config.num_synapses, device=device)
    
    with torch.no_grad():
        for _ in range(samples):
            traj = generate_trajectory(task_type, max_steps=5)
            x = torch.tensor([traj], dtype=torch.long).to(device)
            _, routing_weights = model(x)
            
            # In the last layer's routing, aggregate the indices of the Synapses that were used
            layer_weights = routing_weights[-1][0]  # [seq_len, n_synapses]
            chosen = layer_weights.argmax(dim=-1)
            usage_counts += torch.bincount(chosen, minlength=config.num_synapses)
            
    usage_pct = (usage_counts / usage_counts.sum()).cpu().numpy()
    return usage_pct

usage_treasure = analyze_task_usage("treasure")
usage_escape = analyze_task_usage("escape")

# Compare with a bar chart
plt.figure(figsize=(10, 5))
x = np.arange(config.num_synapses)
width = 0.35

plt.bar(x - width/2, usage_treasure, width, label='Treasure Task', color='gold')
plt.bar(x + width/2, usage_escape, width, label='Escape Task', color='crimson')

plt.ylabel('Usage Percentage')
plt.xlabel('Synapse Index')
plt.title('Synapse Usage Comparison between Tasks')
plt.xticks(x)
plt.legend()
plt.show()

print("[Observation] Bar heights differ across tasks (i.e. different Synapses are being used).")
print("The router assigns the 'policy for escaping the chaser' and the 'policy for approaching the treasure' to different Synapses.")

## 5. Análise de roteamento: Separação de percepção e ação (visualização por token)
Para uma única trajetória de “caça ao tesouro”, usamos um mapa de calor para verificar a qual sinapse cada token (Estado, Recompensa, Ação) foi atribuído.

In [ ]:
model.eval()
traj = generate_trajectory("treasure", max_steps=2)
x = torch.tensor([traj], dtype=torch.long).to(device)

with torch.no_grad():
    _, routing_weights = model(x)

weights = routing_weights[0][0].cpu().numpy()
from src.constants import ID2TOK
labels = [ID2TOK.get(t, str(t)) for t in traj]

plt.figure(figsize=(12, 8))
sns.heatmap(weights, cmap="YlGnBu", yticklabels=labels, annot=False)
plt.title("Token-wise Synapse Routing (Treasure Task)")
plt.xlabel("Synapse Index")
plt.ylabel("Token Type")
plt.show()

print("[Observation] Looking at the labels on the vertical axis, you can see that for each data type")
print("such as 'State (coordinates)', 'Reward', and 'Action', the highlighted Synapses (horizontal axis) are cleanly separated.")